In [1]:
import numpy as np
from collections import Counter
from itertools import combinations, permutations
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)

ld = LammpsData.from_file('bulk_structure/mg21zn25_relaxed.data', atom_style='atomic')
alloy = AseAtomsAdaptor().get_atoms(ld.structure)
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")

Bulk: 276 atoms, Mg126Zn150


In [2]:
slab = surface(alloy, (1, 1, 0), 2, vacuum=8)

sym = np.array(slab.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')
z = slab.get_positions()[:, 2]
thick = z.max() - z.min()

ps = AseAtomsAdaptor().get_structure(slab)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(1,1,0), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(slab)}, Mg: {mg}, Zn: {zn}")
print(f"Thickness: {thick:.1f} A")
print(f"Ratio Zn/Mg: {zn/mg:.4f} (need {25/21:.4f})")
print(f"Stoichiometric: {'YES' if zn*21 == mg*25 else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 552, Mg: 252, Zn: 300
Thickness: 26.1 A
Ratio Zn/Mg: 1.1905 (need 1.1905)
Stoichiometric: YES
Symmetric: False


In [3]:
z = slab.get_positions()[:, 2]
sym = np.array(slab.get_chemical_symbols())
order = np.argsort(z)

z_sorted = np.sort(z)
gaps = np.diff(z_sorted)
tol = max(0.02, min(0.3, np.median(gaps[gaps > 0.01]) / 2))
print(f"Using tolerance: {tol:.3f} A\n")

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

z_min, z_max = z.min(), z.max()
n = len(layers)

print(f"Total atomic layers: {n}\n")
print(f"{'Bot':>7} {'Bot Comp':>10} {'from_bot':>9}  |  {'Top':>7} {'Top Comp':>10} {'from_top':>9} {'Match':>6}")
print("-" * 80)
for i in range(min(15, n//2)):
    bot, top = layers[i], layers[n-1-i]
    zm_b = np.mean([z[j] for j in layer_idx[i]])
    zm_t = np.mean([z[j] for j in layer_idx[n-1-i]])
    
    mg_b, zn_b = bot.get('Mg',0), bot.get('Zn',0)
    mg_t, zn_t = top.get('Mg',0), top.get('Zn',0)
    comp_b = f"Mg{mg_b}Zn{zn_b}" if mg_b and zn_b else (f"Mg{mg_b}" if mg_b else f"Zn{zn_b}")
    comp_t = f"Mg{mg_t}Zn{zn_t}" if mg_t and zn_t else (f"Mg{mg_t}" if mg_t else f"Zn{zn_t}")
    
    match = "YES" if bot == top else "NO <---"
    print(f"{i:>7} {comp_b:>10} {zm_b-z_min:>9.3f}  |  {n-1-i:>7} {comp_t:>10} {z_max-zm_t:>9.3f} {match:>6}")

Using tolerance: 0.088 A

Total atomic layers: 55

    Bot   Bot Comp  from_bot  |      Top   Top Comp  from_top  Match
--------------------------------------------------------------------------------
      0       Zn12     0.007  |       54        Zn6     0.000 NO <---
      1       Zn12     0.795  |       53       Zn12     0.780    YES
      2        Mg6     1.289  |       52        Mg6     1.275    YES
      3   Mg12Zn12     1.520  |       51   Mg12Zn12     1.505    YES
      4    Mg18Zn6     3.043  |       50    Mg18Zn6     3.028    YES
      5       Zn12     3.852  |       49       Zn12     3.837    YES
      6        Mg6     3.996  |       48        Mg6     3.982    YES
      7       Zn12     4.623  |       47       Zn12     4.609    YES
      8        Mg3     5.068  |       46        Mg3     5.054    YES
      9        Zn6     5.345  |       45        Zn6     5.330    YES
     10        Mg6     5.522  |       44        Mg6     5.507    YES
     11        Zn6     5.809  |       4

In [4]:
# Search for symmetric removals
print("Searching for symmetric removals...\n")

for bot_rm in range(0, 6):
    for top_rm in range(0, 6):
        if bot_rm == 0 and top_rm == 0:
            continue
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) == 0:
            continue
        tr = slab[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(1,1,0), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            if slab_tr.is_symmetric():
                removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
                removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                sym_tr = np.array(tr.get_chemical_symbols())
                print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                      f"Mg{sum(sym_tr=='Mg')} Zn{sum(sym_tr=='Zn')}, "
                      f"removed {removed_bot}+{removed_top}={removed_bot+removed_top}")
        except:
            continue

Searching for symmetric removals...

  bot_rm=1, top_rm=1: 534 atoms, Mg252 Zn282, removed 12+6=18
  bot_rm=2, top_rm=2: 510 atoms, Mg252 Zn258, removed 24+18=42
  bot_rm=3, top_rm=3: 498 atoms, Mg240 Zn258, removed 30+24=54
  bot_rm=4, top_rm=4: 450 atoms, Mg216 Zn234, removed 54+48=102
  bot_rm=5, top_rm=5: 402 atoms, Mg180 Zn222, removed 78+72=150


In [5]:
removed_bot = layer_idx[0]
removed_top = layer_idx[n-1]
removed_all = removed_bot + removed_top

keep = []
for j in range(1, n - 1):
    keep.extend(layer_idx[j])

trimmed = slab[keep]
ps_trim = AseAtomsAdaptor().get_structure(trimmed)

sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"SG = {sga.get_space_group_symbol()}")

for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

rem_mg = sum(sym[j] == 'Mg' for j in removed_all)
rem_zn = sum(sym[j] == 'Zn' for j in removed_all)
print(f"\nRemoved: {len(removed_all)} atoms (Mg{rem_mg} Zn{rem_zn})")
print(f"  Bottom: {len(removed_bot)}, Top: {len(removed_top)}")

# Check partners
ps_orig = AseAtomsAdaptor().get_structure(slab)

paired = []
for j in removed_bot:
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    for k in removed_top:
        if np.linalg.norm(slab.positions[k] - inv_cart) < 0.5:
            paired.append((j, k))
            break

used_top = set(k for _, k in paired)
unpaired_bot = [j for j in removed_bot if j not in [b for b, _ in paired]]
unpaired_top = [j for j in removed_top if j not in used_top]

print(f"\nPaired (bot<->top): {len(paired)}")
print(f"Unpaired bottom: {len(unpaired_bot)}")
for j in unpaired_bot:
    pos = slab.positions[j]
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    print(f"  idx={j} {sym[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f}) "
          f"-> inv: ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")
print(f"Unpaired top: {len(unpaired_top)}")

SG = P2/c
Inversion: f -> [0. 0. 0.] - f

Removed: 18 atoms (Mg0 Zn18)
  Bottom: 12, Top: 6

Paired (bot<->top): 6
Unpaired bottom: 6
  idx=240 Zn (15.081, 2.929, 8.000) -> inv: (30.162, 5.859, 34.106)
  idx=100 Zn (15.081, 7.323, 8.000) -> inv: (30.162, 1.465, 34.106)
  idx=190 Zn (30.162, 1.465, 8.000) -> inv: (15.081, 7.323, 34.106)
  idx=49 Zn (30.162, 5.859, 8.000) -> inv: (15.081, 2.929, 34.106)
  idx=2 Zn (45.243, 0.000, 8.000) -> inv: (0.000, 0.000, 34.106)
  idx=139 Zn (45.243, 4.394, 8.000) -> inv: (0.000, 4.394, 34.106)
Unpaired top: 0


In [6]:
sc_fixed = slab.copy()
ps_orig = AseAtomsAdaptor().get_structure(slab)

move_pairs = [
    (49, 240),    # move 49 to inv(240)
    (190, 100),   # move 190 to inv(100)
    (139, 2),     # move 139 to inv(2)
]

for move_idx, keep_idx in move_pairs:
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (-frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    old = sc_fixed.positions[move_idx].copy()
    sc_fixed.positions[move_idx] = inv_cart
    print(f"Moved {move_idx} ({sym[move_idx]}) -> inv({keep_idx}): "
          f"({old[0]:.3f}, {old[1]:.3f}, {old[2]:.3f}) -> "
          f"({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

sym_f = np.array(sc_fixed.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
    miller_index=(1,1,0), oriented_unit_cell=ps_fixed, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"\nAtoms: {len(sc_fixed)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if zn_f*21 == mg_f*25 else 'NO'}")
print(f"Symmetric: {slab_fixed.is_symmetric()}")

Moved 49 (Zn) -> inv(240): (30.162, 5.859, 8.000) -> (30.162, 5.859, 34.106)
Moved 190 (Zn) -> inv(100): (30.162, 1.465, 8.000) -> (30.162, 1.465, 34.106)
Moved 139 (Zn) -> inv(2): (45.243, 4.394, 8.000) -> (0.000, 0.000, 34.106)

Atoms: 552, Mg: 252, Zn: 300
Stoichiometric: YES
Symmetric: True


In [7]:
view(sc_fixed)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [8]:
thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc_fixed)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg21zn25_110_2layers_reconstructed_ase.data")

print(f"Saved: slabs/mg21zn25_110_2layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

Saved: slabs/mg21zn25_110_2layers_reconstructed_ase.data
  Atoms: 552
  Thickness: 26.1 A
  Area: 397.6 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [9]:
#unrelaxed surface energy calculation
E_slab =    -690.62669    # eV
E_bulk_per_fu = -363.4709  / 6  # eV per formula unit = 
n = 552 / 46                # formula units in slab =
A = 397.6             # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -60.578483 eV
n * E_bulk = -726.94180 eV
E_slab - n*E_bulk = 36.31511 eV
S = 0.045668 eV/Ang^2
S = 0.7317 J/m^2


In [11]:
#relaxed surface energy calculation
E_slab_relaxed =  -693.808524190678  # eV
E_bulk_per_fu =  -363.4709  / 6  # eV per formula unit
n = 552 / 46                      # 32 formula units
A = 397.6                  # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.7317
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 33.13328 eV
S relaxed = 0.041667 eV/Ang^2
S relaxed = 0.6676 J/m^2

Unrelaxed S = 0.7317 J/m^2
Relaxed S   = 0.6676 J/m^2
Relaxation energy = 0.0641 J/m^2
